# Создание проекта разметки бабочек

Создадим проект, обучение, экзамен и основной пул.
В обучении и экзамене по 5 фото, в основном пуле — 400 фото и 30 ханипотов.

Рядом с тетрадкой должна лежать папка `platform_task_files`.


In [1]:
%pip install -q "requests>=2.32,<3"



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## API

Токен берём из `YSDA_BUTTERFLIES_TOKEN` или вводим вручную.


In [2]:
import json
import os
from datetime import datetime, timedelta, timezone
from getpass import getpass
from html import escape
from pathlib import Path

import requests

TOKEN = os.environ.get("YSDA_BUTTERFLIES_TOKEN") or getpass("Токен Яндекс Заданий: ")
BASE_URL = "https://tasks.yandex.ru/api/v1"
headers = {
    "Authorization": f"OAuth {TOKEN}",
    "Content-Type": "application/json",
}


def request(method, url, **kwargs):
    response = requests.request(method, url, headers=headers, timeout=60, **kwargs)
    if not response.ok:
        message = response.text.replace(TOKEN, "[скрыто]")[:1500]
        raise RuntimeError(f"{method} {url}: HTTP {response.status_code}: {message}")
    return response.json()


## Настройки

Оплата за страницу. Страницы принимаются автоматически, ошибки на ханипотах ограничивают доступ к проекту.


In [ ]:
DATA_DIR = Path("platform_task_files")
PROJECT_NAME = "Разметка бабочек — проверка IoU"
BBOX_PARTICIPATION_SKILL_ID = "21403"

IOU_THRESHOLD = 0.75
TRAINING_PER_PAGE = 5
EXAM_PER_PAGE = 5
TRAINING_PASS_PERCENT = 0
EXAM_PASS_PERCENT = 80

REAL_PER_PAGE = 2
GOLDEN_PER_PAGE = 1
REWARD_PER_PAGE = 1.0
OVERLAP = 1
TIME_LIMIT_SECONDS = 1800

QUALITY_WINDOW = 5
QUALITY_PASS_PERCENT = 80
QUALITY_BAN_DAYS = 7
FAST_PAGE_SECONDS = 9
FAST_PAGES_LIMIT = 2


In [4]:
will_expire = (datetime.now(timezone.utc) + timedelta(days=30)).strftime(
    "%Y-%m-%dT%H:%M:%S"
)


## Данные

Читаем задания из JSON. В соседних TSV лежат те же данные для загрузки через сайт.
Эталон есть только в обучении, экзамене и ханипотах. `category` оставляем пустым.


In [5]:
def read_tasks(filename):
    return json.loads((DATA_DIR / filename).read_text(encoding="utf-8"))


training_tasks = read_tasks("training_5.json")
exam_tasks = read_tasks("exam_5.json")
honeypots = read_tasks("honeypots_30.json")
main_tasks = read_tasks("annotation_400.json")
print(
    {
        "Обучение": len(training_tasks),
        "Экзамен": len(exam_tasks),
        "Ханипоты": len(honeypots),
        "Разметка": len(main_tasks),
    }
)


{'Обучение': 5, 'Экзамен': 5, 'Ханипоты': 30, 'Разметка': 400}


## Загрузка заданий

Отправляем по 50 заданий за запрос.


In [6]:
def upload_tasks(pool_id, tasks):
    for start in range(0, len(tasks), 50):
        batch = [dict(task, pool_id=pool_id) for task in tasks[start : start + 50]]
        response = request(
            "POST",
            BASE_URL + "/tasks",
            json=batch,
            params={
                "async_mode": "false",
                "skip_invalid_items": "false",
                "allow_defaults": "true",
            },
        )
        if response.get("validation_errors"):
            raise RuntimeError(
                json.dumps(response["validation_errors"], ensure_ascii=False)
            )
    print("Пул", pool_id, "— загружено заданий:", len(tasks))


## Интерфейс

Фото в одной колонке. Q — выбрать рамку, E — нарисовать, X — удалить выбранную.
Режим переключается одним нажатием.


### HTML карточки


In [7]:
MARKUP = r"""
<div class="bbox">
    <div class="bbox-view">
        <iframe class="bbox-editor" title="Редактор рамок" allow="fullscreen"></iframe>
    </div>
</div>
"""


### Размеры и расположение фото


In [8]:
STYLES = r"""
body:has(.bbox) {
    margin: 0;
    padding: 0;
}
.task-suite:has(.bbox) {
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 12px;
    padding: 8px;
    box-sizing: border-box;
}
.task-suite .task:has(.bbox) {
    display: block;
    width: 100%;
    max-width: 1042px;
    min-width: 0;
    margin: 0;
    padding: 10px;
    box-sizing: border-box;
    flex-shrink: 0;
}
.bbox {
    width: 1000px;
    max-width: 100%;
    min-width: 0;
    margin: 0 auto;
    color: #222;
    line-height: 1.4;
}
.bbox-view {
    height: clamp(360px, calc(100vh - 160px), 1000px);
    min-width: 0;
}
.bbox-editor {
    display: block;
    width: 100%;
    height: 100%;
    border: 0;
}
.bbox-editor.bbox-fullscreen {
    position: fixed;
    inset: 0;
    width: 100vw;
    height: 100vh;
    z-index: 100000;
    background: white;
}
"""


### Редактор рамок

Используем стандартный редактор, оставляем прямоугольники.


In [9]:
TB_TEMPLATE = {
    "config": {
        "view": {
            "type": "layout.bars",
            "barBefore": {
                "type": "view.markdown",
                "content": "**Разметьте всех взрослых бабочек, включая мотыльков и моль.**",
            },
            "content": {
                "type": "field.image-annotation",
                "image": {
                    "type": "data.input",
                    "path": "image",
                },
                "data": {
                    "type": "data.output",
                    "path": "result",
                },
                "fullHeight": False,
                "minWidth": 150,
                "shapes": {"rectangle": True},
                "ratio": {
                    "type": "data.input",
                    "path": "editor_ratio",
                },
            },
        },
        "plugins": [
            {
                "type": "plugin.field.image-annotation.hotkeys",
                "modes": {
                    "select": "q",
                    "rectangle": "e",
                },
                "confirm": "a",
                "cancel": "esc",
            }
        ],
    },
    "lock": {
        "core": "1.14.0",
        "layout.bars": "1.0.5",
        "view.markdown": "1.0.8",
        "field.image-annotation": "1.5.0",
        "plugin.field.image-annotation.hotkeys": "1.0.6",
    },
}


### Загрузчик редактора

HTML подключает стандартный загрузчик и стили внутри iframe.


In [10]:
RUNTIME_HTML = r"""
<!doctype html>
<html>
    <head>
        <meta charset="utf-8" />
        <title>Template</title>
        <script
            defer="defer"
            src="https://yastat.net/s3/tb/launcher.embed/_/embed.b3746f87b77bc931aa99.js"
        ></script>
        <link
            href="https://yastat.net/s3/tb/launcher.embed/_/embed.css"
            rel="stylesheet"
        />
    </head>
    <body>
        <script>
            window.TB_REGISTRY_URL = "https://tb.yandex.net/registry2/";
            window.TB_REGISTRY_PRERELEASE_TAG = "";
        </script>
    </body>
</html>
"""


### Разрешения iframe

CSP задаёт, откуда редактор может загружать файлы и куда отправлять запросы.


In [11]:
RUNTIME_CSP = r"""
default-src 'none';
img-src https: http: data: https://api-maps.yandex.ru https://*.api-maps.yandex.ru
    https://*.maps.yandex.net https://yastatic.net;
media-src http: https: data:;
frame-src https:;
style-src 'unsafe-inline' 'self' blob: yastat.net yastatic.net
    https://api-maps.yandex.ru https://*.api-maps.yandex.ru;
script-src 'unsafe-eval' 'unsafe-inline' 'self' mc.yandex.ru yastat.net
    https://yastat.net yastatic.net api-maps.yandex.ru https://*.api-maps.yandex.ru
    *.maps.yandex.net https://suggest-maps.yandex.ru;
connect-src 'self' yastat.net prestable.tb.yandex.net testing.tb.yandex.net
    tb.yandex.net *.samsara.yandex-team.ru samsara.yandex-team.ru search.yandex-team.ru
    test.nirvana.yandex-team.ru nirvana.yandex-team.ru https://*.yandex.ru
    https://*.yandex.net https://*.yandex-team.ru https://*.yandex.com
    https://*.yandex.com.tr https://*.yandex.uz https://eaas.in.alice.yandex.net
    https://api-maps.yandex.ru https://*.api-maps.yandex.ru
    https://suggest-maps.yandex.ru https://search-maps.yandex.ru
    https://api.routing.yandex.net https://yandex.ru/clck/click;
worker-src blob: data: 'self' https://api-maps.yandex.ru https://*.api-maps.yandex.ru
    https://yastatic.net;
child-src blob: 'self';
font-src data: yastatic.net yastat.net;
"""


### Удаление выбранной рамки


In [12]:
DELETE_X_JS = r"""
window.addEventListener(
    "keydown",
    function (event) {
        if (
            event.code !== "KeyX" ||
            event.ctrlKey ||
            event.metaKey ||
            event.altKey ||
            event.repeat
        )
            return;
        if (
            event.target.closest &&
            event.target.closest('input,textarea,[contenteditable="true"]')
        )
            return;
        var button = Array.from(document.querySelectorAll("button")).find(
            function (el) {
                return (
                    el.textContent.trim() === "Удалить прямоугольник" &&
                    !el.disabled &&
                    el.getClientRects().length
                );
            },
        );
        if (!button) return;
        event.preventDefault();
        event.stopImmediatePropagation();
        button.click();
    },
    true,
);
"""


### Проверка по IoU

Сопоставляем рамки с эталоном без повторений. Пропуски, лишние рамки или IoU ниже порога дают `bbox_correct = false`.


In [13]:
IOU_JS = r"""
function validRectangle(box) {
    return (
        box &&
        box.shape === "rectangle" &&
        [box.left, box.top, box.width, box.height].every(function (v) {
            return typeof v === "number" && Number.isFinite(v);
        }) &&
        box.left >= 0 &&
        box.top >= 0 &&
        box.width > 0 &&
        box.height > 0 &&
        box.left + box.width <= 1.000001 &&
        box.top + box.height <= 1.000001
    );
}

function boxIoU(a, b) {
    var width = Math.max(0, Math.min(a[2], b[2]) - Math.max(a[0], b[0]));
    var height = Math.max(0, Math.min(a[3], b[3]) - Math.max(a[1], b[1]));
    var intersection = width * height;
    var union =
        (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - intersection;
    return union > 0 ? intersection / union : 0;
}

function bboxCorrect(boxes, reference) {
    if (
        !Array.isArray(boxes) ||
        !boxes.every(validRectangle) ||
        !Array.isArray(reference)
    )
        return false;
    if (boxes.length !== reference.length) return false;
    if (
        !reference.every(function (b) {
            return (
                b &&
                [b.left, b.top, b.width, b.height].every(function (v) {
                    return typeof v === "number" && Number.isFinite(v);
                }) &&
                b.left >= 0 &&
                b.top >= 0 &&
                b.width > 0 &&
                b.height > 0 &&
                b.left + b.width <= 1.000001 &&
                b.top + b.height <= 1.000001
            );
        })
    )
        return false;
    var actual = boxes.map(function (b) {
        return [
            b.left,
            b.top,
            Math.min(1, b.left + b.width),
            Math.min(1, b.top + b.height),
        ];
    });
    var expected = reference.map(function (b) {
        return [
            b.left,
            b.top,
            Math.min(1, b.left + b.width),
            Math.min(1, b.top + b.height),
        ];
    });
    var matches = expected.map(function () {
        return -1;
    });
    function match(i, seen) {
        for (var j = 0; j < expected.length; j++) {
            if (seen[j] || boxIoU(actual[i], expected[j]) + 1e-12 < IOU_THRESHOLD)
                continue;
            seen[j] = true;
            if (matches[j] === -1 || match(matches[j], seen)) {
                matches[j] = i;
                return true;
            }
        }
        return false;
    }
    return actual.every(function (_, i) {
        return match(
            i,
            expected.map(function () {
                return false;
            }),
        );
    });
}

function updateAnswer(task, boxes) {
    task.setSolutionOutputValue("result", boxes);
    var input = task.getTask().input_values;
    if (Object.prototype.hasOwnProperty.call(input, "reference_boxes")) {
        task.setSolutionOutputValue(
            "bbox_correct",
            bboxCorrect(boxes, input.reference_boxes),
        );
    } else if (task.getSolution().output_values.bbox_correct !== undefined) {
        task.setSolutionOutputValue("bbox_correct", undefined);
    }
}
"""


### Связь с редактором

Передаём фото и получаем рамки. Эталон остаётся в родительской странице.


In [14]:
EDITOR_JS = r"""
function createTbEditor(frame, input, output, readOnly, onChange) {
    var origin = "null";
    function onMessage(event) {
        if (
            event.origin !== origin ||
            event.source !== frame.contentWindow ||
            !event.data
        )
            return;
        var message = event.data;
        if (message.__tbIframeAction !== "request") return;
        var data;
        if (message.method === "embedReady") {
            data = {
                features: Object.assign({}, message.payload.features, {
                    submit: "",
                    submitByEnter: false,
                    locales: ["ru"],
                    locale: "ru",
                    isReadOnly: Boolean(readOnly),
                    computeLock: false,
                }),
                templateState: {
                    config: TB_TEMPLATE.config,
                    lock: TB_TEMPLATE.lock,
                    input: {
                        image: input.image,
                        category: input.category,
                        editor_ratio: [
                            Math.max(1, frame.clientWidth - 57),
                            Math.max(1, frame.clientHeight - 40),
                        ],
                    },
                    output: { result: output.result || [] },
                    uploads: [],
                },
            };
        } else if (
            ["embedValueChanged", "embedSubmitV2"].indexOf(message.method) >= 0
        ) {
            var value = message.payload.output;
            if (
                value &&
                typeof value === "object" &&
                (value.result === undefined || Array.isArray(value.result))
            ) {
                onChange(value.result || []);
            }
        } else if (message.method === "requestEmbedStateChange") {
            frame.classList.toggle(
                "bbox-fullscreen",
                Boolean(message.payload.isFullscreen),
            );
            data = { isFullscreen: Boolean(message.payload.isFullscreen) };
        }
        frame.contentWindow.postMessage(
            { __tbIframeAction: "response", id: message.id, ok: true, data: data },
            "*",
        );
    }
    window.addEventListener("message", onMessage);
    frame.setAttribute("sandbox", "allow-scripts allow-modals");
    frame.srcdoc = TB_EMBED_HTML;
    return {
        destroy: function () {
            window.removeEventListener("message", onMessage);
        },
    };
}
"""


### Сохранение ответа

При изменении рамок обновляем ответ. На эталонных фото записываем ещё и `bbox_correct`.


In [15]:
TASK_JS = r"""
exports.Task = function (options) {
    TolokaHandlebarsTask.call(this, options);
    if (!Array.isArray(this.getSolution().output_values.result))
        this.setSolutionOutputValue("result", []);
};
exports.Task.prototype = Object.create(TolokaHandlebarsTask.prototype);
exports.Task.prototype.constructor = exports.Task;
exports.Task.prototype.onRender = function () {
    var task = this;
    this._editor = createTbEditor(
        this.getDOMElement().querySelector(".bbox-editor"),
        this.getTask().input_values,
        this.getSolution().output_values,
        this.getWorkspaceOptions().isReadOnly,
        function (boxes) {
            updateAnswer(task, boxes);
        },
    );
};
exports.Task.prototype.onDestroy = function () {
    if (this._editor) this._editor.destroy();
};
"""


### Инструкция исполнителю


In [16]:
INSTRUCTIONS = r"""
<h2>Выделите всех взрослых бабочек на фотографии</h2>
<p>
    Обведите каждую взрослую бабочку отдельной прямоугольной рамкой. Размечайте дневных
    и ночных бабочек, включая мотыльков и моль.
</p>
<h3>Правила разметки</h3>
<ul>
    <li>
        Найдите всех бабочек, включая маленьких, частично закрытых и находящихся у края
        фотографии.
    </li>
    <li>
        Плотно охватите все видимые части: крылья, тело, голову, усики и лапки. Не
        обрезайте их и не добавляйте большой запас фона.
    </li>
    <li>
        Для частично закрытой бабочки выделяйте видимые части одной рамкой. Не
        достраивайте скрытые части; фон между видимыми частями допустим.
    </li>
    <li>Если бабочка выходит за кадр, рамка заканчивается на границе фотографии.</li>
    <li>
        Одна бабочка — одна рамка. Рамки разных бабочек могут пересекаться. Не
        объединяйте несколько бабочек одной рамкой и не дублируйте рамки.
    </li>
    <li>
        НЕ выделяйте других насекомых, гусениц, куколок, яйца, растения, рисунки,
        отражения бабочек и отдельные оторванные крылья.
    </li>
</ul>
<h3>Как выполнить задание?</h3>
<p>
    Осмотрите всю фотографию. Используйте инструмент «Прямоугольник». Один раз нажмите Q
    или кнопку со стрелкой, чтобы перейти в режим выбора уже выделенной рамки. Один раз
    нажмите E или кнопку с прямоугольником, чтобы перейти в режим выделения новой рамки.
    Клавиши и кнопки переключают режим; удерживать их не нужно. Чтобы удалить одну
    рамку, включите режим Q, щёлкните по нужной рамке и один раз нажмите X. Кнопка
    корзины удаляет все рамки.
</p>
<p>
    Если взрослых бабочек нет, оставьте фотографию без рамок. Дополнительный ответ
    выбирать не нужно.
</p>
<p>
    Проблема загрузки и сомнение не означают отсутствие бабочек. Просматривайте каждую
    фотографию внимательно.
</p>
<p>
    Все выданные фотографии показаны на странице. Осмотрите каждую фотографию и
    проверьте рамки перед отправкой.
</p>
"""


### Сборка интерфейса


In [17]:
editor_addons = f"""
<base href="https://tb.yandex.net/">
<meta http-equiv="Content-Security-Policy" content="{escape(RUNTIME_CSP, quote=True)}">
<script>
{DELETE_X_JS}
</script>
"""
editor_html = RUNTIME_HTML.replace("<head>", "<head>\n" + editor_addons)

interface_js = "\n".join(
    [
        "var TB_TEMPLATE = " + json.dumps(TB_TEMPLATE, ensure_ascii=False) + ";",
        "var TB_EMBED_HTML = " + json.dumps(editor_html, ensure_ascii=False) + ";",
        f"var IOU_THRESHOLD = {IOU_THRESHOLD};",
        IOU_JS,
        EDITOR_JS,
        TASK_JS,
    ]
)

view_specification = {
    "type": "classic",
    "assets": {"script_urls": ["$TOLOKA_ASSETS/js/toloka-handlebars-templates.js"]},
    "markup": MARKUP,
    "styles": STYLES,
    "script": interface_js,
    "settings": {
        "showSkip": True,
        "showTimer": True,
        "showFinish": True,
        "showSubmit": True,
        "showFullscreen": True,
        "showInstructions": True,
    },
}


## Проект

Зададим поля задания и создадим новый проект.


### Поля задания


In [ ]:
input_specification = {
    "photo_id": {"type": "string", "required": True, "hidden": False},
    "image": {"type": "url", "required": True, "hidden": False},
    "category": {"type": "string", "required": False, "hidden": False, "min_length": 0},
    "reference_boxes": {"type": "json", "required": False, "hidden": False},
}

output_specification = {
    "result": {"type": "array_json", "required": False, "hidden": False},
    "bbox_correct": {"type": "boolean", "required": False, "hidden": False},
}


### Навык участия

После первой принятой страницы выдаём навык участия. Он исключает исполнителя из review-проекта.


In [ ]:
participation_rule = {
    "collector_config": {"type": "ANSWER_COUNT"},
    "rules": [
        {
            "conditions": [
                {
                    "key": "assignments_accepted_count",
                    "operator": "GTE",
                    "value": 1,
                }
            ],
            "action": {
                "type": "SET_SKILL",
                "parameters": {
                    "skill_id": BBOX_PARTICIPATION_SKILL_ID,
                    "skill_value": 1,
                },
            },
        }
    ],
}


### Создание проекта


In [ ]:
project_payload = {
    "public_name": PROJECT_NAME,
    "public_description": "Выделите всех взрослых бабочек, включая мотыльков и моль, прямоугольными рамками.",
    "public_instructions": INSTRUCTIONS,
    "assignments_issuing_type": "AUTOMATED",
    "assignments_automerge_enabled": False,
    "task_spec": {
        "input_spec": input_specification,
        "output_spec": output_specification,
        "view_spec": view_specification,
    },
}
project_payload["quality_control"] = {"configs": [participation_rule]}
project = request("POST", BASE_URL + "/projects", json=project_payload)
project_id = str(project["id"])
print(f"https://tasks.yandex.ru/requester/project/{project_id}")


## Навык экзамена

В нём хранится процент правильных ответов. Ханипоты этот навык не меняют.


In [19]:
skill_payload = {
    "name": f"Бабочки: экзамен проекта {project_id}",
    "hidden": True,
}
exam_skill = request("POST", BASE_URL + "/skills", json=skill_payload)
exam_skill_id = str(exam_skill["id"])


## Обучение

Пять фото на одной бесплатной странице. При ошибке показываем подсказку и даём исправить рамки.
Порог обучения — 0%, допуск определяет экзамен.


In [ ]:
training_payload = {
    "project_id": project_id,
    "private_name": "Обучение: 5 фото",
    "inherited_instructions": True,
    "may_contain_adult_content": False,
    "assignment_max_duration_seconds": TIME_LIMIT_SECONDS,
    "mix_tasks_in_creation_order": True,
    "shuffle_tasks_in_task_suite": False,
    "training_tasks_in_task_suite_count": TRAINING_PER_PAGE,
    "task_suites_required_to_pass": 1,
}
training = request("POST", BASE_URL + "/trainings", json=training_payload)
training_pool_id = str(training["id"])


### Навык обучения

Платформа создаёт его сама. Сделаем его скрытым.


In [20]:
training = request("GET", f"{BASE_URL}/trainings/{training_pool_id}")
training_skill_id = str(training["skill_id"])
training_skill = request("GET", f"{BASE_URL}/skills/{training_skill_id}")
request(
    "PUT",
    f"{BASE_URL}/skills/{training_skill_id}",
    json={
        "name": training_skill["name"],
        "hidden": True,
    },
)


{'id': '21399',
 'name': '11769 Разметка бабочек — проверка IoU - training',
 'public_name': {'EN': '11769 Разметка бабочек — проверка IoU - training'},
 'hidden': True,
 'deprecated': False,
 'type': 'TRAINING',
 'owner': {'id': 'ae8ff80dc4e53068894abc39f77490df',
  'myself': True,
  'company_id': '151'},
 'training': True,
 'created': '2026-11-15T17:22:22.376',
 'global': False}

In [21]:
upload_tasks(training_pool_id, training_tasks)


Пул 6866201 — загружено заданий: 5


## Экзамен

Пять фото на одной бесплатной странице. Для допуска нужно не меньше 80% — четыре верных ответа.


In [22]:
exam_pool_payload = {
    "project_id": project_id,
    "private_name": "Экзамен: 5 фото",
    "type": "REGULAR",
    "may_contain_adult_content": False,
    "reward_per_assignment": 0.0,
    "assignment_max_duration_seconds": TIME_LIMIT_SECONDS,
    "auto_accept_solutions": True,
    "will_expire": will_expire,
    "defaults": {
        "default_overlap_for_new_tasks": 10,
        "default_overlap_for_new_task_suites": 10,
    },
    "mixer_config": {
        "real_tasks_count": 0,
        "golden_tasks_count": EXAM_PER_PAGE,
        "training_tasks_count": 0,
        "min_real_tasks_count": 0,
        "min_golden_tasks_count": EXAM_PER_PAGE,
        "force_last_assignment": False,
    },
    "filter": {
        "and": [
            {
                "category": "profile",
                "key": "languages",
                "operator": "IN",
                "value": "RU",
            },
            {
                "category": "computed",
                "key": "client_type",
                "operator": "EQ",
                "value": "BROWSER",
            },
        ]
    },
}


### Результат экзамена

После пяти ответов записываем процент верных в навык экзамена.


In [23]:
exam_pool_payload["quality_control"] = {
    "training_requirement": {
        "training_pool_id": training_pool_id,
        "training_passing_skill_value": TRAINING_PASS_PERCENT,
    },
    "configs": [
        {
            "collector_config": {
                "type": "GOLDEN_SET",
                "parameters": {},
            },
            "rules": [
                {
                    "conditions": [
                        {
                            "key": "golden_set_answers_count",
                            "operator": "GTE",
                            "value": len(exam_tasks),
                        }
                    ],
                    "action": {
                        "type": "SET_SKILL_FROM_OUTPUT_FIELD",
                        "parameters": {
                            "skill_id": exam_skill_id,
                            "from_field": "golden_set_correct_answers_rate",
                        },
                    },
                }
            ],
        }
    ],
}
exam_pool = request("POST", BASE_URL + "/pools", json=exam_pool_payload)
exam_pool_id = str(exam_pool["id"])


In [24]:
upload_tasks(exam_pool_id, exam_tasks)


Пул 6866202 — загружено заданий: 5


## Основной пул

Два обычных фото и один ханипот на странице за 1 ₽. Перекрытие — 1.


In [25]:
main_pool_payload = {
    "project_id": project_id,
    "private_name": "Разметка: 400 фото + 30 ханипотов",
    "type": "REGULAR",
    "may_contain_adult_content": False,
    "will_expire": will_expire,
    "reward_per_assignment": REWARD_PER_PAGE,
    "assignment_max_duration_seconds": TIME_LIMIT_SECONDS,
    "auto_accept_solutions": True,
    "defaults": {
        "default_overlap_for_new_tasks": OVERLAP,
        "default_overlap_for_new_task_suites": OVERLAP,
    },
    "mixer_config": {
        "real_tasks_count": REAL_PER_PAGE,
        "golden_tasks_count": GOLDEN_PER_PAGE,
        "training_tasks_count": 0,
        "min_real_tasks_count": REAL_PER_PAGE,
        "min_golden_tasks_count": GOLDEN_PER_PAGE,
        "force_last_assignment": False,
    },
    "filter": {
        "and": [
            {
                "category": "profile",
                "key": "languages",
                "operator": "IN",
                "value": "RU",
            },
            {
                "category": "computed",
                "key": "client_type",
                "operator": "EQ",
                "value": "BROWSER",
            },
            {
                "category": "skill",
                "key": exam_skill_id,
                "operator": "GTE",
                "value": EXAM_PASS_PERCENT,
            },
        ]
    },
}


### Слишком быстрые ответы

Две страницы быстрее 9 секунд закрывают доступ к проекту постоянно.


In [ ]:
speed_rule = {
    "collector_config": {
        "type": "ASSIGNMENT_SUBMIT_TIME",
        "parameters": {"fast_submit_threshold_seconds": FAST_PAGE_SECONDS},
    },
    "rules": [
        {
            "conditions": [
                {
                    "key": "fast_submitted_count",
                    "operator": "GTE",
                    "value": FAST_PAGES_LIMIT,
                }
            ],
            "action": {
                "type": "RESTRICTION_V2",
                "parameters": {
                    "scope": "PROJECT",
                    "duration_unit": "PERMANENT",
                },
            },
        }
    ],
}


### Ошибки на ханипотах

Точность ниже 80% на последних пяти ханипотах закрывает доступ на 7 дней.


In [ ]:
quality_rule = {
    "collector_config": {
        "type": "GOLDEN_SET",
        "parameters": {"history_size": QUALITY_WINDOW},
    },
    "rules": [
        {
            "conditions": [
                {
                    "key": "golden_set_answers_count",
                    "operator": "GTE",
                    "value": QUALITY_WINDOW,
                },
                {
                    "key": "golden_set_correct_answers_rate",
                    "operator": "LT",
                    "value": QUALITY_PASS_PERCENT,
                },
            ],
            "action": {
                "type": "RESTRICTION_V2",
                "parameters": {
                    "scope": "PROJECT",
                    "duration": QUALITY_BAN_DAYS,
                    "duration_unit": "DAYS",
                },
            },
        }
    ],
}


### Создание пула


In [26]:
main_pool_payload["quality_control"] = {"configs": [speed_rule, quality_rule]}
main_pool = request("POST", BASE_URL + "/pools", json=main_pool_payload)
main_pool_id = str(main_pool["id"])


### Обычные задания

400 фото без эталона.


In [27]:
upload_tasks(main_pool_id, main_tasks)


Пул 6866203 — загружено заданий: 400


### Ханипоты

Добавляем в тот же пул. Платформа сама собирает страницы из двух обычных фото и одного ханипота.


In [28]:
upload_tasks(main_pool_id, honeypots)


Пул 6866203 — загружено заданий: 30
